## Предсказание стоимости жилья

В проекте вам нужно обучить модель линейной регрессии на данных о жилье в Калифорнии в 1990 году. На основе данных нужно предсказать медианную стоимость дома в жилом массиве. Обучите модель и сделайте предсказания на тестовой выборке. Для оценки качества модели используйте метрики RMSE, MAE и R2.

# Задача

**обучить модель линейной регрессии** на данных о жилье в Калифорнии в 1990 году, **предсказать медианную стоимость дома в жилом массиве** — median_house_value. 

**метрики** оценки качества модели: RMSE, MAE и R2.

<div class="alert alert-secondary" style="background-color:#D9EEE1;color:black;">

## Описание данных

- longitude — долгота;
- latitude — широта;
- housing_median_age — медианный возраст жителей жилого массива;
- total_rooms — общее количество комнат в домах жилого массива;
- total_bedrooms — общее количество спален в домах жилого массива;
- population — количество человек, которые проживают в жилом массиве;
- households — количество домовладений в жилом массиве;
- median_income — медианный доход жителей жилого массива;
- ocean_proximity — близость к океану.
- **median_house_value** — медианная стоимость дома в жилом массиве;
    - таргет


In [219]:
!pip install pyspark -q

In [220]:
import pandas as pd
import numpy as np
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F
from pyspark.sql.functions import isnan, isnull, when, count, col
import os
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler,\
                                StandardScaler
from pyspark.ml import Pipeline

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator


In [221]:
def correct_path (csv_name):
    if not os.path.exists(csv_name):
        return '/datasets/' + csv_name
    return csv_name

# Подготовка данных

In [222]:
path = './california_housing_w_features'

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

Инициализируйте локальную Spark-сессию.

In [223]:
spark = SparkSession.builder \
                    .master("local") \
                    .appName("EDA California Housing") \
                    .getOrCreate()

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

Прочитайте содержимое файла /datasets/housing.csv.

In [224]:
df_housing = spark.read.load(correct_path('housing.csv'), 
                                            format="csv", sep=",", inferSchema=True, header="true")
df_housing.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

Выведите типы данных колонок датасета. Используйте методы pySpark.

In [225]:
display(pd.DataFrame(df_housing.dtypes, columns=['column', 'type']).head(10))

,column,type
0,longitude,double
1,latitude,double
2,housing_median_age,double
3,total_rooms,double
4,total_bedrooms,double
5,population,double
6,households,double
7,median_income,double
8,median_house_value,double
9,ocean_proximity,string


In [226]:
display(df_housing.count())

20640

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

Выполните предобработку данных:

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

- Исследуйте данные на наличие пропусков и заполните их, выбрав значения по своему усмотрению.

In [227]:
for column in df_housing.columns:
    check_col = isnan(F.col(column)) | isnull(F.col(column))
    print(column, df_housing.filter(check_col).count())

longitude 0
latitude 0
housing_median_age 0
total_rooms 0
total_bedrooms 207
population 0
households 0
median_income 0
median_house_value 0
ocean_proximity 0


In [228]:
avg_bedrooms = df_housing.agg(F.avg('total_bedrooms')).first()['avg(total_bedrooms)']
df_housing = df_housing.na.fill(avg_bedrooms)

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

- Преобразуйте колонку с категориальными значениями техникой One hot encoding.

Также масштабируем и количественные признаки.

In [229]:
target = 'median_house_value'
categorical_cols = ['ocean_proximity']
categorical_cols_ind = [c+'_idx' for c in categorical_cols]
categorical_cols_ohe = [c+'_ohe' for c in categorical_cols]

numerical_cols = [c for c in df_housing.columns \
                   if c not in categorical_cols and c != target]

numerical_cols_scl = [c+'_scl' for c in categorical_cols]

In [230]:
df = df_housing

indexer = StringIndexer(inputCols=categorical_cols, 
                        outputCols=categorical_cols_ind)

encoder = OneHotEncoder(inputCols=categorical_cols_ind, 
                        outputCols=categorical_cols_ohe)

numerical_assembler = VectorAssembler(inputCols=numerical_cols,
                                        outputCol="numerical_features")

standardScaler = StandardScaler(inputCol='numerical_features',
                                outputCol='numerical_features_scaled')

final_features = categorical_cols_ohe + ['numerical_features_scaled', target]

pipeline = Pipeline(stages=[indexer, encoder, 
                            numerical_assembler, standardScaler])
df = pipeline.fit(df).transform(df)

In [231]:
all_features = ['ocean_proximity_ohe','numerical_features_scaled']

final_assembler = VectorAssembler(inputCols=all_features, 
                                  outputCol="features") 
df = final_assembler.transform(df)

In [232]:
df.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+-------------------+-------------------+--------------------+-------------------------+--------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|ocean_proximity_idx|ocean_proximity_ohe|  numerical_features|numerical_features_scaled|            features|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+-------------------+-------------------+--------------------+-------------------------+--------------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|                3.0|      (4,[3],[1.0])|[-122.23,37.88,41...|     [-61.007269596069...|[0.0,0.0,0.0,1.0,...|
|  -122.22|   37

# Обучение моделей

In [233]:
train_data, test_data = df.randomSplit([.8,.2], seed=42)
display(train_data.count(), test_data.count())

16560

4080

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

Постройте модель линейной регрессии на разных наборах данных:
- используя все данные из файла;

Для построения модели используйте оценщик LinearRegression из библиотеки MLlib.

In [234]:
lr_1 = LinearRegression(labelCol=target, featuresCol='features',
                        predictionCol="prediction")

model_1 = lr_1.fit(train_data) 

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

Постройте модель линейной регрессии на разных наборах данных:
- используя только числовые переменные, исключив категориальные.

In [235]:
lr_2 = LinearRegression(labelCol=target, featuresCol='numerical_features_scaled',
                        predictionCol="prediction")

model_2 = lr_2.fit(train_data) 

# Анализ результатов

<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">


Сравните результаты работы линейной регрессии на двух наборах данных по метрикам RMSE, MAE и R2.

In [236]:
def evaluate (predictions, test_data, predictionCol, labelCol):
    evaluator = RegressionEvaluator(predictionCol=predictionCol,
                                        labelCol=labelCol)
    rmse = int(evaluator.evaluate(predictions, {evaluator.metricName: "rmse"}))
    mae = int(evaluator.evaluate(predictions, {evaluator.metricName: "mae"}))
    r2 = round(evaluator.evaluate(predictions, {evaluator.metricName: "r2"}), 2)
    return rmse, mae, r2

In [237]:
predictions_1 = model_1.transform(test_data)
rmse_1, mae_1, r2_1 = evaluate (predictions_1, test_data, 
                                'prediction', target)

In [238]:
predictions_2 = model_2.transform(test_data)
rmse_2, mae_2, r2_2 = evaluate (predictions_2, test_data, 
                                'prediction', target)

In [239]:
result_df = pd.DataFrame({'rmse': [rmse_1, rmse_2],
                          'mae': [mae_1, mae_2],
                          'r2': [r2_1, r2_2],}, 
                          index=['модель со всеми признаками',
                                 'модель только с числовыми признаками'])
display(result_df)

,rmse,mae,r2
модель со всеми признаками,70781,50855,0.64
модель только с числовыми признаками,71782,51787,0.63


<div class="alert alert-secondary" style="background-color:#DCDCDC;color:black;">

Сделайте выводы.

- средствами pyspark:
    - создана локальная сессия
    - загружены данные
    - удалены 207 пропусков в прищнаке total_bedrooms
    - произведено кодирование текстовых признаков
    - произведено масштабирование количественных признаков
    - обучены две модели линейной регрессии:
        - певая модель: обучена на всех признаках
        - вторая модель: обучена только на количественных признаках
        - метрики качества (представлены в таблице выше) - лучше у первой модели